# routes/preview

> Route handlers for media file serving and preview panel rendering

In [ ]:
#| default_exp routes.preview

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import mimetypes
from typing import Dict, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter
from starlette.responses import FileResponse, Response

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls
from cjm_transcription_source_select.routes.core import (
    get_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.preview_panel import render_preview_panel

## Media Source Handler

Serves local media files for HTML5 `<audio>` and `<video>` elements. Accepts an absolute file path as a query parameter and returns the file with the appropriate MIME type.

In [ ]:
#| export
def _handle_media_src(
    path: str,  # Absolute path to the media file
) -> Response:  # FileResponse or 404
    """Serve a local media file with appropriate MIME type."""
    if not path or not Path(path).is_file():
        return Response(status_code=404)

    media_type, _ = mimetypes.guess_type(path)
    return FileResponse(str(path), media_type=media_type)

## Preview Handler

Renders the preview panel for a selected file. Looks up the file in the current selection state to get metadata.

In [ ]:
#| export
def _handle_preview(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to preview
):  # Rendered preview panel
    """Render the preview panel for a selected file."""
    if not path:
        return render_preview_panel(media_src_url=urls.media_src)

    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])

    # Find the file in the selection
    selected_file = None
    for f in selected_files:
        if f["path"] == path:
            selected_file = f
            break

    if selected_file is None:
        return render_preview_panel(media_src_url=urls.media_src)

    return render_preview_panel(
        selected_file=selected_file,
        media_src_url=urls.media_src,
        is_open=True,
    )

## Router Initialization

In [ ]:
#| export
def init_preview_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    prefix: str = "/preview",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize preview routes for media serving and preview panel rendering."""
    router = APIRouter(prefix=prefix)

    @router
    def media_src(path: str = ""):
        """Serve a local media file."""
        return _handle_media_src(path)

    @router
    def preview(sess, path: str = ""):
        """Render the preview panel for a file."""
        return _handle_preview(state_store, workflow_id, urls, sess, path)

    routes = {
        "media_src": media_src,
        "preview": preview,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()